# A photonic band gap: the band diagram of a square lattice of alumina rods

**Question.** Alumina rods ($\varepsilon_r = 8.9$, radius 2 mm) stand in air on a square lattice of
period $a = 10$ mm, infinitely long. For which frequencies can no wave cross the
crystal in its plane, in either polarization? How fast does light cross it at the
frequencies it does pass? And where in the cell would more permittivity widen the gap?

A wave in a periodic crystal is a Bloch wave: its field repeats from cell to cell up to
a phase set by a wave vector $\mathbf{k}$. At each $\mathbf{k}$ the cell has a ladder of
resonant frequencies. Following one rung as $\mathbf{k}$ moves gives a band, and the
bands along a path through the high-symmetry points of the Brillouin zone make the
band diagram. A frequency range that holds no band at any $\mathbf{k}$ is a band gap.

**Assumptions.**

- The rods are infinitely long, so the crystal is two-dimensional. gfem solves it as a
  {term}`slab model` one element thick: PEC faces on the two ends give the Ez
  polarization (E along the rods), and natural faces give the Hz polarization (H along
  the rods). The textbook calls Ez the TM polarization. This page says Ez and Hz.
- Alumina is lossless and non-dispersive: $\varepsilon_r = 8.9$ at every frequency.
- Each rod is a regular 24-sided polygon with the area of the disk. A straight-sided mesh
  facets a round rod, and the faceting is nearly all of the error of the bands. The equal area
  keeps the cell's mean permittivity exact.
- The crystal is infinite: one cell with Bloch phases stands for all of it.
- The gap is read from a band diagram along the edge of the irreducible zone, where the
  band edges of this crystal sit. A path that misses them reads a gap that is too wide.
  The page checks this against the textbook value and a finite-difference derivative.
- The page does not run an optimization loop. Over permittivity the gap only grows, so
  there is no interior optimum. Over the rod radius a loop needs a shape morph or a remesh
  per step. The sensitivity map of the last section answers the design question
  directly.

**Conventions.** Fields vary as $e^{+i\omega t}$. Across a periodic pair of translation
$\mathbf{a}$, a Bloch mode obeys $\mathbf{E}(\mathbf{r}+\mathbf{a}) =
e^{-i\mathbf{k}\cdot\mathbf{a}}\,\mathbf{E}(\mathbf{r})$. Wave vectors are Cartesian, in
rad/m. Bands are numbered from 0, the lowest: the textbook's bands 1 and 2 are bands 0
and 1 here. Frequencies are plotted as $\omega a/2\pi c$, which does not depend on the
scale of the lattice: the same figure holds for any $a$ with the same permittivities. See
{doc}`../conventions`.

In [ ]:
import time
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Circle
from matplotlib.tri import Triangulation
from scipy.constants import c

import gfem

mm = 1e-3
A = 10 * mm  # lattice constant
R_ROD = 0.2 * A
EPS_ROD = 8.9
tic = time.perf_counter()

## The crystal

{download}`data/rods.msh` is one cell of the lattice as a 2D section, a plane of triangles
with the rod as region `"rod"`, the air as `"air"`, and the four edges of the square as
`"x_min"`, `"x_max"`, `"y_min"` and `"y_max"`. It is built by `tools/rods_mesh.py`, which
meshes one eighth of the square and mirrors it, so the mesh has the symmetry of the
square: it keeps the symmetry of the crystal, which a band diagram is about. Its triangles
are at most 1.25 mm on a side.
{py:meth}`gfem.Mesh.from_gmsh_cross_section` extrudes it into a slab one element thick.
Each pair of opposite edges is one {py:class}`gfem.Periodic` boundary.

Between PEC plates the field is normal to the plates and uniform through the thickness:
this is the Ez polarization. Between natural faces H is normal to the plates instead:
the Hz polarization. For rods that are uniform along z the two families are exact. The
next family, with a field that varies along z, starts at about 1.3 $c/a$, above everything
this page uses.

In [ ]:
mesh = gfem.Mesh.from_gmsh_cross_section(Path("data/rods.msh"), length_unit="mm")
print(mesh)


def crystal(mesh, *, ez):
    boundaries = [
        gfem.Periodic(("x_min", "x_max")),
        gfem.Periodic(("y_min", "y_max")),
    ]
    if ez:
        boundaries += [gfem.PEC("front"), gfem.PEC("back")]
    return gfem.Model(
        mesh,
        materials={"rod": gfem.Material(eps_r=EPS_ROD), "air": gfem.Material()},
        boundaries=boundaries,
    )


ez = crystal(mesh, ez=True)
hz = crystal(mesh, ez=False)
print(
    "translations (mm):",
    np.round(np.array(ez.periodic_translations)[:, :2] / mm, 3).tolist(),
)
print(
    "|b| a / 2 pi:",
    np.round(np.linalg.norm(ez.reciprocal_lattice, axis=1) * A / (2 * np.pi), 3),
)

The model finds its two translations from the mesh, 10 mm along $x$ and along $y$.
{py:attr}`gfem.Model.reciprocal_lattice` holds the vectors $\mathbf{b}_i$ with
$\mathbf{b}_i\cdot\mathbf{a}_j = 2\pi\delta_{ij}$: here $|\mathbf{b}_i| = 2\pi/a$. The
figure shows the section's triangles, with the rod shaded.

In [ ]:
front = mesh.elements("front")  # the triangles of one face of the slab
triangles = mesh.surface_elements[front]
region = [next(v for v in pair if v) for pair in mesh.adjacency("front")]
in_rod = np.array([r == "rod" for r in region])
xy = mesh.nodes[:, :2] / mm
triang = Triangulation(xy[:, 0], xy[:, 1], triangles)

fig, ax = plt.subplots(figsize=(3.6, 3.6))
ax.tripcolor(
    triang,
    in_rod.astype(float),
    cmap="Greys",
    vmin=0,
    vmax=2.5,
    edgecolors="k",
    linewidth=0.3,
)
ax.set_aspect("equal")
ax.set_xlabel("x (mm)")
ax.set_ylabel("y (mm)")
ax.set_title(f"one cell, {len(triangles)} triangles")
plt.show()

## A path through the Brillouin zone

The bands are a function of $\mathbf{k}$ in the first Brillouin zone, a square of side
$2\pi/a$. The square's symmetry (the point group C4v) relates the zone's eight octants, so the
bands over the whole zone are the bands over one octant, a triangle with corners Γ
(the centre), X (the middle of an edge) and M (the corner). The extrema of the bands usually
sit on that triangle's edge, so the diagram follows it: Γ to X to M and back to Γ.
{py:func}`gfem.wave_vector_path` names these {term}`high-symmetry points <high-symmetry point>` in
reduced coordinates of the {term}`reciprocal lattice`, X at $(1/2, 0)$ and M at $(1/2, 1/2)$, and
spaces `n_points` wave vectors along the path by length, with every named point among
them. Each wave vector is one solve, so the cost grows linearly with `n_points`.

In [ ]:
path = gfem.wave_vector_path(ez, ["Γ", "X", "M", "Γ"], n_points=31)
i_x, i_m = path.index("X"), path.index("M")
print(path)
print("X is wave vector", i_x, "and M is wave vector", i_m)

## The bands

{py:func}`gfem.band_diagram` solves the eigenmode problem at each wave vector of the path and
orders the modes by frequency into {term}`bands <band>`. It uses one fixed target
frequency. If the target lies below half the lowest frequency of the highest band you ask
for, the modes found are the lowest ones, and band $n$ is the $n$-th lowest
({term}`anchored band window`). Here the target is $0.25\,c/a$ for four bands, and band 3
never goes below about $0.63\,c/a$.

At Γ the solve drops the fields that stay at zero frequency, and the diagram puts the
{term}`static band` back in at 0. `keep_fields` keeps the field at the wave vectors named,
X and M, for the next section; no other wave vector keeps its field.

In [ ]:
options = gfem.SolverOptions(order=2)
TARGET = 0.25 * c / A  # Hz, below half of band 3 everywhere on the path

ez_bands = gfem.band_diagram(
    ez, path, target=TARGET, n_bands=4, keep_fields=[i_x, i_m], options=options
)
hz_bands = gfem.band_diagram(hz, path, target=TARGET, n_bands=4, options=options)
ez_f = np.asarray(ez_bands.frequency) * A / c  # omega a / 2 pi c
hz_f = np.asarray(hz_bands.frequency) * A / c
print(ez_bands)
print(hz_bands)

In [ ]:
lo, hi = ez_f[:, 0].max(), ez_f[:, 1].min()  # the Ez gap, between bands 0 and 1

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.axhspan(lo, hi, color="0.88", zorder=0)
for band in range(4):
    ax.plot(path.distance, ez_f[:, band], color="C0", label="Ez" if band == 0 else None)
    ax.plot(
        path.distance,
        hz_f[:, band],
        color="C3",
        ls="--",
        label="Hz" if band == 0 else None,
    )
ax.set_xticks(*path.ticks)
ax.set_xlim(path.distance[0], path.distance[-1])
ax.set_ylim(0, 0.9)
ax.set_xlabel("wave vector")
ax.set_ylabel(r"$\omega a / 2\pi c$")
ax.secondary_yaxis(
    "right", functions=(lambda f: f * c / A / 1e9, lambda g: g * 1e9 * A / c)
).set_ylabel("frequency (GHz)")
ax.legend(loc="upper left")
ax.set_title("bands of the rod lattice, Ez gap shaded")
plt.show()

The shaded band is the Ez gap, between the top of band 0 and the bottom of band 1. No Ez
wave in this frequency range crosses the crystal in any direction in its plane. The Hz bands
(dashed) have no such gap.

### Against the textbook

Joannopoulos et al., *Photonic Crystals: Molding the Flow of Light*, 2nd ed., ch. 5,
give a square lattice of $\varepsilon = 8.9$ rods of radius $0.2a$ an Ez gap of
31.4 % between the first two bands. The gap is measured as the gap-midgap ratio
$\Delta\omega/\omega_m = 2(\omega_1^\mathrm{min} - \omega_0^\mathrm{max})/(\omega_1^\mathrm{min} +
\omega_0^\mathrm{max})$, the gap's width over its centre frequency. This is the {term}`band gap`
as a ratio that does not depend on the scale of the lattice. The cell below reads it
off the bands, and finds where the two band edges sit.

In [ ]:
labels_at = dict(zip(path.label_index, path.labels, strict=True))
gap_ratio = 2 * (hi - lo) / (hi + lo)
edge_k = (labels_at[int(np.argmax(ez_f[:, 0]))], labels_at[int(np.argmin(ez_f[:, 1]))])
print(f"Ez gap: {lo:.4f} to {hi:.4f} c/a, width over centre {100 * gap_ratio:.1f} %")
print(f"band 0 tops out at {edge_k[0]}, band 1 bottoms out at {edge_k[1]}")

hz_overlap = (
    hz_f[:, 0].max() - hz_f[:, 1].min()
)  # > 0: Hz bands 0 and 1 overlap, no Hz gap
print(
    f"Hz band 0 reaches {hz_f[:, 0].max():.4f}, "
    f"Hz band 1 starts at {hz_f[:, 1].min():.4f}: overlap {hz_overlap:.4f} c/a"
)
print(f"Hz band 0 covers the Ez gap: {hz_f[:, 0].max() > hi}")

doublet_split = abs(ez_f[i_m, 2] - ez_f[i_m, 1]) / ez_f[i_m, 1]
print(
    f"Ez bands 1 and 2 at M: {ez_f[i_m, 1]:.5f}, "
    f"{ez_f[i_m, 2]:.5f}, split {doublet_split:.1e}"
)
print(
    "anchored at every wave vector:",
    bool(ez_bands.anchored.all() and hz_bands.anchored.all()),
)
print("lost bands:", bool(ez_bands.lost.any() or hz_bands.lost.any()))

The Ez gap is 31.4 %, as published, with band 0 topping out at M and band 1 bottoming out at X.
Both edges are simple bands, which matters in the last section. In the Hz polarization the
first band reaches above the bottom of the second, so there is no Hz gap, and
the Hz band 0 covers the whole Ez gap: the crystal has no gap common to both polarizations.
Isolated high-permittivity rods favour Ez gaps (Joannopoulos, ch. 5).

At M, Ez bands 1 and 2 are a degenerate pair by the square's symmetry. They come out split
by about $10^{-5}$ of their frequency. That is the mesh's residual asymmetry, not
physics: each square of the section is cut into tetrahedra by a rule that has no mirror
symmetry. The split is below the error of the bands and invisible in the figure.
`anchored` is True at every wave vector and no band is `lost`: the path
is fine enough for the connections to hold, and the target holds the bands' numbers.

## The modes at the gap's edges

Why do these two bands sit where they do? The band-0 mode at M is the dielectric band, the
top of the lowest band. The band-1 mode at X is the air band, the bottom of the next. The
fields are on the section's mid-plane, and $|E_z|^2$ is normalized to its maximum. The
`rod` participation is the share of the mode's electric energy stored in the rods.

In [ ]:
n = 120
xs = (np.arange(n) + 0.5) / n * A
gx, gy = np.meshgrid(xs, xs)
points = np.column_stack(
    [gx.ravel(), gy.ravel(), np.full(gx.size, mesh.section_thickness / 2)]
)


def energy_map(index, band):
    e = np.asarray(ez_bands.e_field(index, band).at(points))
    w = np.abs(e[:, 2]) ** 2  # the field is along z
    return (w / w.max()).reshape(n, n)


p_rod = np.asarray(ez_bands.bulk_participation["rod"])
modes = [
    ("band 0 at M, the dielectric band", i_m, 0),
    ("band 1 at X, the air band", i_x, 1),
]
fig, axes = plt.subplots(1, 2, figsize=(7, 3.4), layout="constrained")
for ax, (title, index, band) in zip(axes, modes, strict=True):
    image = ax.pcolormesh(
        xs / mm,
        xs / mm,
        energy_map(index, band),
        cmap="gray",
        vmin=0,
        vmax=1,
        shading="auto",
    )
    ax.add_patch(
        Circle((A / 2 / mm, A / 2 / mm), R_ROD / mm, fill=False, ec="w", lw=0.8)
    )
    ax.set_aspect("equal")
    ax.set_xlabel("x (mm)")
    ax.set_title(
        f"{title}\n{100 * p_rod[index, band]:.0f} % of the energy in the rod",
        fontsize=8,
    )
axes[0].set_ylabel("y (mm)")
fig.colorbar(image, ax=axes, label=r"$|E_z|^2$ (normalized)", shrink=0.8)
plt.show()

The variational theorem (Joannopoulos, ch. 2) says a mode lowers its frequency by putting
its energy where the permittivity is high. The lower mode concentrates its energy in the
rods, and the upper mode, kept orthogonal to it, has a node there and lives in the air.
The gap is the price of that difference: the same cell holds both modes at the same
$\mathbf{k}$-space distance from Γ, and their frequencies differ by the permittivity the two
fields see.

## How fast light crosses the crystal

The band group velocity $\mathbf{v}_g = \partial\omega/\partial\mathbf{k}$ is the speed at which a wave
packet of that band moves. It is the exact derivative of the discrete band, given at no
extra solve: `ez_bands.v_g` is in m/s, with shape (wave vectors, bands, 3). Its component
along the path is the slope of the band in the figure above. The figure compares it with
the slope of the plotted bands, taken by centred differences. The markers sit on the
curves.

In [ ]:
vg = np.asarray(ez_bands.v_g.real)  # (31, 4, 3), m/s
step = np.diff(path.wave_vectors, axis=0)
tangent = np.vstack([step, step[-1:]])  # the segment each wave vector starts
tangent /= np.linalg.norm(tangent, axis=1, keepdims=True)
along = np.einsum("kbd,kd->kb", vg, tangent) / c

d = path.distance
inner = np.array([i for i in range(1, len(d) - 1) if i not in path.label_index])
slope = (
    2
    * np.pi
    * (ez_f[inner + 1] - ez_f[inner - 1])
    / (A * (d[inner + 1] - d[inner - 1])[:, None])
)

fig, ax = plt.subplots(figsize=(6.5, 3.6))
for band in range(4):
    ax.plot(d, along[:, band], color=f"C{band}", label=f"band {band}")
    ax.plot(d[inner], slope[:, band], ".", color=f"C{band}", ms=4)
ax.axhline(0, color="0.6", lw=0.5)
ax.set_xticks(*path.ticks)
ax.set_xlim(d[0], d[-1])
ax.set_xlabel("wave vector")
ax.set_ylabel(r"$\mathbf{v}_g\cdot\hat{\mathbf{t}}\,/\,c$")
ax.legend(ncol=4, fontsize=8, loc="lower left")
ax.set_title("group velocity along the path (curves), slope of the bands (dots)")
plt.show()

In [ ]:
k_small = 0.01 * 2 * np.pi / A
long_wave = gfem.solve(
    ez,
    gfem.Eigenmode(0.01 * c / A, n_modes=1, wave_vector=(k_small, 0.0, 0.0)),
    options=options,
)
v_long = float(long_wave.v_g[0, 0].real) / c
eps_mean = 1 + (EPS_ROD - 1) * np.pi * 0.2**2  # the mean permittivity of the cell
print(
    f"long-wave limit: v_g = {v_long:.4f} c, "
    f"1 / sqrt(mean eps) = {1 / np.sqrt(eps_mean):.4f}"
)

v_edge = max(np.linalg.norm(vg[i_m, 0]), np.linalg.norm(vg[i_x, 1])) / c
print(f"|v_g| at the band edges (band 0 at M, band 1 at X): {v_edge:.1e} c")

slope_error = np.nanmax(np.abs(slope - along[inner]))
print(f"centred differences against v_g along the path: {slope_error:.1e} c")

slowed = c / np.linalg.norm(vg[i_x - 1, 1])
print(f"band 1 one step before X: light slowed {slowed:.0f}-fold")

Three checks. In the long-wave limit a wave sees the crystal as a uniform medium of the
mean permittivity $\langle\varepsilon\rangle = 1 + 7.9\pi(0.2)^2$, so
$|\mathbf{v}_g| \to c/\sqrt{\langle\varepsilon\rangle} = 0.7084\,c$, which the equal-area rod
makes exact. (At $k = 0.01\cdot 2\pi/a$ the band has already bent a little.) At a band
edge a simple band is a standing wave: $\mathbf{v}_g = 0$ at X and at M. And the group
velocity matches the slope of the bands inside each segment.

Light slows near a band edge. One step before X the air band's $|\mathbf{v}_g|$ is the number
above divided into $c$. This ratio $c/|\mathbf{v}_g|$ is the factor by which the packet
travels slower than light in vacuum. It is not the group index of a waveguide mode, which
gfem computes for ports and names differently. `v_g` is NaN on the static band at Γ and
on bands that touch with different slopes.

## Where more permittivity widens the gap

The gap-midgap ratio $R$ is a function of the permittivity of every element of the cell,
through the bands. `ratio` builds the model with the rod's and the air's permittivities as
per-element arrays ({py:class}`gfem.MaterialField`; a constant {py:class}`gfem.Material` is
static), runs the whole band diagram, and returns $R$ from `bands.omega`.
{py:func}`jax.value_and_grad` gives $\partial R/\partial\varepsilon_e$ for every element at
once. The gradient costs a backward pass per wave vector and no extra eigenvalue solve: it is
Hellmann-Feynman, as in {doc}`../recipes/eigenmode-gradient`.

In [ ]:
def ratio(eps_rod, eps_air):
    model = ez.with_material("rod", gfem.MaterialField(eps_r=eps_rod))
    model = model.with_material("air", gfem.MaterialField(eps_r=eps_air))
    bands = gfem.band_diagram(model, path, target=TARGET, n_bands=4, options=options)
    w = bands.omega.real
    top_of_band_0, bottom_of_band_1 = jnp.max(w[:, 0]), jnp.min(w[:, 1])
    return 2 * (bottom_of_band_1 - top_of_band_0) / (bottom_of_band_1 + top_of_band_0)


n_rod, n_air = len(mesh.elements("rod")), len(mesh.elements("air"))
t0 = time.perf_counter()
value, (g_rod, g_air) = jax.value_and_grad(ratio, argnums=(0, 1))(
    jnp.full(n_rod, EPS_ROD), jnp.ones(n_air)
)
g_rod, g_air = np.asarray(g_rod), np.asarray(g_air)
print(
    f"gap-midgap ratio {100 * float(value):.3f} %, "
    f"with its gradient in {time.perf_counter() - t0:.0f} s"
)
print(f"dR/d eps_rod, the sum over the rod:  {g_rod.sum():+.4f}")
print(f"dR/d eps_air, the sum over the air:  {g_air.sum():+.4f}")

Raising the permittivity of the rod by one unit widens the gap by 0.032 of its width over its
centre. Raising the permittivity of the air between the rods by one unit narrows it nine times
as fast. Filling the air between the rods, with a plastic foam say, closes the gap: the
background matters more than the rods. The map shows where. Each triangle's value is the
gradient of its three tetrahedra, divided by the triangle's share of the cell's area, so that
the map integrates to the two numbers above. Red is large in magnitude: the rod's
elements are mostly positive and the air's mostly negative.

In [ ]:
tets = mesh.volume_elements
centroids = mesh.nodes[tets].mean(axis=1)
which = triang.get_trifinder()(centroids[:, 0] / mm, centroids[:, 1] / mm)
np.testing.assert_array_less(-1, which)
per_tet = np.zeros(len(tets))
per_tet[mesh.elements("rod")] = g_rod
per_tet[mesh.elements("air")] = g_air
per_triangle = np.bincount(which, weights=per_tet, minlength=len(triangles))

corners = xy[triangles]
area = (
    0.5
    * np.abs(np.cross(corners[:, 1] - corners[:, 0], corners[:, 2] - corners[:, 0]))
    * mm**2
)
density = per_triangle / (area / A**2)  # dR / d eps per unit share of the cell
print(
    f"the map integrates to {np.sum(density * area / A**2):+.4f}, "
    f"the sum of the two sums above is {g_rod.sum() + g_air.sum():+.4f}"
)

fig, ax = plt.subplots(figsize=(3.9, 3.6))
image = ax.tripcolor(triang, np.abs(density), cmap="gfem_sensitivity", shading="flat")
ax.add_patch(
    Circle((A / 2 / mm, A / 2 / mm), R_ROD / mm, fill=False, ec="k", lw=0.8, ls="--")
)
ax.set_aspect("equal")
ax.set_xlabel("x (mm)")
ax.set_ylabel("y (mm)")
ax.set_title("the gap's sensitivity to permittivity", fontsize=9)
fig.colorbar(image, label=r"$|\partial R/\partial\varepsilon|$ per unit area share")
plt.show()

The map reads like the energy maps above: the gap is sensitive where the two band-edge
modes keep their energy. Two checks follow. Perturbation theory (Joannopoulos, ch. 2) gives
the shift of a band by a change of permittivity in the rod, from the energy the mode stores
there, $\partial\omega_n/\partial\varepsilon_\mathrm{rod} = -\omega_n\,p_{n,\mathrm{rod}} /
(2\varepsilon_\mathrm{rod})$, with $p_{n,\mathrm{rod}}$ the `bulk_participation` of the mode.
For the lossless discrete problem it is exact, so its combination into $\partial R/\partial
\varepsilon_\mathrm{rod}$ must equal the sum of the gradient over the rod. And a central
difference on the rod's scalar permittivity, run on the two wave vectors that hold the band
edges (X and M, from the section above), must give the same number.

In [ ]:
w = np.asarray(ez_bands.omega.real)
w_lo, w_hi = w[i_m, 0], w[i_x, 1]  # the edges: band 0 at M, band 1 at X
d_lo = -w_lo * p_rod[i_m, 0] / (2 * EPS_ROD)
d_hi = -w_hi * p_rod[i_x, 1] / (2 * EPS_ROD)
d_pt = 4 * (w_lo * d_hi - w_hi * d_lo) / (w_hi + w_lo) ** 2
d_rod = g_rod.sum()
perturbation_error = abs(d_pt - d_rod) / abs(d_rod)
print(
    f"perturbation theory {d_pt:+.6f}, "
    f"gradient {d_rod:+.6f}, relative difference {perturbation_error:.1e}"
)

edges = gfem.wave_vector_path(ez, ["X", "M"], n_points=2)


def ratio_at_edges(eps_rod):
    bands = gfem.band_diagram(
        ez.with_material("rod", gfem.Material(eps_r=eps_rod)),
        edges,
        target=TARGET,
        n_bands=4,
        options=options,
    )
    f = np.asarray(bands.frequency)
    return 2 * (f[:, 1].min() - f[:, 0].max()) / (f[:, 1].min() + f[:, 0].max())


h = 1e-4
fd = (ratio_at_edges(EPS_ROD + h) - ratio_at_edges(EPS_ROD - h)) / (2 * h)
fd_error = abs(fd - d_rod) / abs(fd)
print(
    f"central difference {fd:+.6f}, "
    f"gradient {d_rod:+.6f}, relative difference {fd_error:.1e}"
)

A gap edge that falls on one member of a degenerate pair, at Γ or M for instance, raises
{py:class}`gfem.DegenerateModeError` under {py:func}`jax.grad`: the derivative of one member of a pair is
not defined. Take the mean of the pair there. The edges of this gap are simple bands, so
the gradient is defined.

## A finer mesh

`data/rods_fine.msh` halves the triangles' sides and meshes the rod as a 48-gon. The Ez bands
at X and M, on the two-point path, give the gap again.

In [ ]:
fine = gfem.Mesh.from_gmsh_cross_section(Path("data/rods_fine.msh"), length_unit="mm")
ez_fine = crystal(fine, ez=True)
fine_bands = gfem.band_diagram(
    ez_fine,
    gfem.wave_vector_path(ez_fine, ["X", "M"], n_points=2),
    target=TARGET,
    n_bands=4,
    options=options,
)
f = np.asarray(fine_bands.frequency) * A / c
gap_fine = 2 * (f[:, 1].min() - f[:, 0].max()) / (f[:, 1].min() + f[:, 0].max())
delta_mesh = abs(gap_fine - gap_ratio)
print(
    f"Ez gap on the finer mesh: {100 * gap_fine:.3f} %, coarse mesh "
    f"{100 * gap_ratio:.3f} %: differ by {delta_mesh:.1e}"
)

The gap moves by about $10^{-5}$ of its centre frequency: the mesh of the page resolves it.

## Limits

- The crystal is two-dimensional. A crystal slab of finite height has a light cone and leaky
  bands, and a projected band diagram: not covered here.
- The rod is a polygon of the disk's area, which keeps its corners. The material is lossless
  and non-dispersive.
- `gfem.band_diagram` runs on one process, not under MPI.
- `band_diagram` runs under {py:func}`jax.grad`, not inside {py:func}`jax.jit` or
  {py:func}`jax.vmap`: loop over designs in Python.
- `v_g` is not differentiable, and there are no gradients with respect to $\mathbf{k}$.
- Bands come in frequency order with connections by field overlap. A group of bands the overlap
  cannot tell apart is flagged (`connection_by_overlap`), not resolved. A path with too few wave
  vectors flags a {term}`lost band`.
- A three-dimensional lattice takes its high-symmetry points from the lattice's name
  or from `named_points` ({doc}`../recipes/periodic-cell`).

## Next

- {doc}`../recipes/periodic-cell`: other lattices, grids of wave vectors, the group velocity.
- {doc}`../recipes/eigenmode-gradient`: gradients of a single mode's frequency.
- {doc}`metagrating`: a periodic cell lit by a plane wave.
- {py:func}`gfem.band_diagram`, {py:func}`gfem.wave_vector_path` and
  {py:mod}`gfem.conventions`.

The numbers this page relies on, checked: the Ez gap is $31.4 \pm 0.1$ %, with its edges at
M and X; the Hz bands 0 and 1 overlap; the doublet at M is split by less than $10^{-4}$; every wave
vector is anchored and no band is lost; the long-wave $|\mathbf{v}_g|$ is $0.7084\,c$; the band
edges are standing waves; the group velocity matches the slope of the bands; the gap's
gradient matches perturbation theory and a central difference; and the finer mesh
leaves the gap where it was.

In [ ]:
np.testing.assert_allclose(gap_ratio, 0.314, atol=1e-3)
np.testing.assert_equal(edge_k, ("M", "X"))
np.testing.assert_array_less(0, hz_overlap)
np.testing.assert_array_less(doublet_split, 1e-4)
np.testing.assert_equal(bool(ez_bands.anchored.all() and hz_bands.anchored.all()), True)
np.testing.assert_equal(bool(ez_bands.lost.any() or hz_bands.lost.any()), False)
np.testing.assert_allclose(v_long, 0.7084, atol=1e-3)
np.testing.assert_array_less(v_edge, 1e-3)
np.testing.assert_array_less(slope_error, 5e-2)
np.testing.assert_array_less(perturbation_error, 1e-8)
np.testing.assert_array_less(fd_error, 1e-6)
np.testing.assert_array_less(delta_mesh, 1e-4)
print(f"page run in {time.perf_counter() - tic:.0f} s")